# CartPole MPR Reservoir RL Scratchpad

This is the CartPole RL notebook using the same style of MPR/rate reservoir dynamics as the MNIST scratchpad: each cell has rate `R`, voltage `V`, input current `I`, and heterogeneous `Delta`, `Eta`, `J`, and `current_decay` parameters.

The task is still Gymnasium `CartPole-v1`. The reservoir is fixed; a linear actor and linear critic are trained from reservoir rate features. This tests whether the earlier biologically flavored dynamics can serve as an RL reservoir.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, replace
import threading
import time

import gymnasium as gym
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from scipy import sparse
from scipy.sparse.linalg import eigs
from tqdm.auto import tqdm

print(f"gymnasium {gym.__version__}; numpy {np.__version__}")


## Configuration

The reservoir weights are fixed. The only trained parameters are the linear actor and critic readouts.


In [ ]:
@dataclass
class ReservoirConfig:
    n_reservoir: int = 256
    input_dim: int = 10
    rec_fan_in: int = 24
    spectral_radius: float = 0.85
    input_scale: float = 1.20
    input_gain: float = 1.0
    seed: int = 1

    # MPR-like rate/voltage/current dynamics, matching the MNIST scratchpad form.
    dt: float = 0.01
    substeps_per_env_step: int = 1
    Delta: float = 1.0
    Eta: float = -2.5
    J: float = 0.0
    current_decay: float = 0.20
    randomize_cell_dynamics: bool = True
    Delta_std: float = 0.15
    Eta_std: float = 0.35
    J_std: float = 0.0
    current_decay_std: float = 0.03
    Delta_clip: tuple[float, float] = (0.05, 4.0)
    Eta_clip: tuple[float, float] = (-8.0, 2.0)
    J_clip: tuple[float, float] = (-3.0, 3.0)
    current_decay_clip: tuple[float, float] = (0.0, 0.95)
    r_clip: tuple[float, float] = (0.0, 20.0)
    v_clip: tuple[float, float] = (-50.0, 50.0)
    i_clip: tuple[float, float] = (-50.0, 50.0)
    rate_feature_scale: float = 1.0


@dataclass
class RLConfig:
    env_id: str = "CartPole-v1"
    seed: int = 0
    train_episodes: int = 1500
    eval_episodes: int = 50
    eval_every: int = 500
    max_steps: int = 500
    # Continued MPR training is much steadier with larger on-policy batches and PPO-style clipping.
    gamma: float = 0.97
    actor_lr: float = 5e-1
    critic_lr: float = 2e-2
    entropy_beta: float = 0.0
    l2: float = 1e-5
    reward_scale: float = 0.01
    normalize_advantage: bool = True
    batch_episodes: int = 32
    max_grad_norm: float = 5.0
    advantage_clip: float = 5.0
    max_policy_logit_step: float = 0.20
    entropy_floor: float = 0.45
    entropy_recovery_beta: float = 0.02
    ppo_clip: float = 0.20
    ppo_epochs: int = 4
    target_kl: float = 0.03
    line_search: bool = True
    line_search_backtracks: int = 8
    line_search_decay: float = 0.5
    armijo_c1: float = 1e-4


RESERVOIR_CONFIG = ReservoirConfig()
RL_CONFIG = RLConfig()


## Helpers

The CartPole observation encoder exposes normalized physics variables plus previous action and normalized episode time.


In [ ]:
def moving_average(values, window: int = 50):
    values = np.asarray(values, dtype=np.float32)
    if len(values) == 0:
        return values
    window = int(max(1, min(window, len(values))))
    kernel = np.ones(window, dtype=np.float32) / window
    prefix = np.full(window - 1, np.nan, dtype=np.float32)
    return np.concatenate([prefix, np.convolve(values, kernel, mode="valid")])


def softmax(logits):
    logits = np.asarray(logits, dtype=np.float32)
    logits = logits - logits.max(axis=-1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=-1, keepdims=True)


def make_env(config, seed: int | None = None):
    env = gym.make(config.env_id)
    if seed is not None:
        env.reset(seed=int(seed))
        env.action_space.seed(int(seed) + 1)
    return env


def encode_cartpole_observation(env, observation, previous_action: float = 0.0, step_fraction: float = 0.0):
    x, x_dot, theta, theta_dot = [float(v) for v in observation]
    theta_limit = float(env.unwrapped.theta_threshold_radians)
    x_limit = float(env.unwrapped.x_threshold)
    return np.array(
        [
            np.clip(x / x_limit, -2.0, 2.0),
            np.clip(x_dot / 3.0, -2.0, 2.0),
            np.clip(theta / theta_limit, -2.0, 2.0),
            np.clip(theta_dot / 3.5, -2.0, 2.0),
            np.sin(theta),
            np.cos(theta),
            float(previous_action),
            float(step_fraction),
            np.clip(abs(theta) / theta_limit, 0.0, 2.0),
            1.0,
        ],
        dtype=np.float32,
    )


## MPR Rate Reservoir

This cell model mirrors the MNIST notebook: recurrent and input currents drive an MPR-like rate/voltage system stepped with RK4. The readout uses rates only, matching the MNIST probe convention.


In [ ]:
class Reservoir:
    frame_title = "MPR reservoir rate"

    def __init__(self, config: ReservoirConfig):
        self.config = config
        self.rng = np.random.default_rng(config.seed)
        self.W_rec = self._make_recurrent_matrix()
        self.W_in = self.rng.normal(
            0.0,
            config.input_scale / np.sqrt(config.input_dim),
            size=(config.n_reservoir, config.input_dim),
        ).astype(np.float32)
        self._build_cell_dynamics()
        self.reset()

    @property
    def n_features(self):
        # Match the MNIST readout convention: use rate features, not voltage.
        return self.config.n_reservoir + self.config.input_dim + 1

    def _make_recurrent_matrix(self):
        cfg = self.config
        rows, cols, data = [], [], []
        for row in range(cfg.n_reservoir):
            presynaptic = self.rng.choice(cfg.n_reservoir, size=cfg.rec_fan_in, replace=False)
            rows.extend([row] * cfg.rec_fan_in)
            cols.extend(presynaptic.tolist())
            data.extend(self.rng.normal(0.0, 1.0 / np.sqrt(cfg.rec_fan_in), size=cfg.rec_fan_in).tolist())
        W = sparse.csr_matrix((np.asarray(data, dtype=np.float32), (rows, cols)), shape=(cfg.n_reservoir, cfg.n_reservoir))
        try:
            rho = float(max(abs(eigs(W.astype(np.float64), k=1, which="LM", return_eigenvectors=False))))
        except Exception:
            rho = 1.0
        return (W * (cfg.spectral_radius / (rho + 1e-6))).tocsr()

    def _sample_cell_parameter(self, mean, std, clip):
        cfg = self.config
        if (not cfg.randomize_cell_dynamics) or std == 0:
            values = np.full(cfg.n_reservoir, mean, dtype=np.float32)
        else:
            values = self.rng.normal(mean, std, size=cfg.n_reservoir).astype(np.float32)
        return np.clip(values, clip[0], clip[1]).astype(np.float32)

    def _build_cell_dynamics(self):
        cfg = self.config
        self.Delta = self._sample_cell_parameter(cfg.Delta, cfg.Delta_std, cfg.Delta_clip)
        self.Eta = self._sample_cell_parameter(cfg.Eta, cfg.Eta_std, cfg.Eta_clip)
        self.J = self._sample_cell_parameter(cfg.J, cfg.J_std, cfg.J_clip)
        self.current_decay = self._sample_cell_parameter(cfg.current_decay, cfg.current_decay_std, cfg.current_decay_clip)

    def reset(self):
        n = self.config.n_reservoir
        self.R = np.zeros(n, dtype=np.float32)
        self.V = np.zeros(n, dtype=np.float32)
        self.I = np.zeros(n, dtype=np.float32)

    def _rk4_step(self):
        cfg = self.config
        R, V, I = self.R, self.V, self.I
        dt = cfg.dt
        pi = np.pi
        pi2 = pi * pi

        def derivative(rate, voltage):
            dR = self.Delta / pi + 2.0 * rate * voltage
            dV = voltage * voltage + self.Eta + self.J * rate + I - pi2 * rate * rate
            return dR, dV

        k1R, k1V = derivative(R, V)
        k2R, k2V = derivative(R + 0.5 * dt * k1R, V + 0.5 * dt * k1V)
        k3R, k3V = derivative(R + 0.5 * dt * k2R, V + 0.5 * dt * k2V)
        k4R, k4V = derivative(R + dt * k3R, V + dt * k3V)
        self.R = np.clip(R + dt / 6.0 * (k1R + 2.0 * k2R + 2.0 * k3R + k4R), *cfg.r_clip).astype(np.float32)
        self.V = np.clip(V + dt / 6.0 * (k1V + 2.0 * k2V + 2.0 * k3V + k4V), *cfg.v_clip).astype(np.float32)

    def step(self, input_vector):
        cfg = self.config
        u = np.asarray(input_vector, dtype=np.float32)
        for _ in range(cfg.substeps_per_env_step):
            self.I = self.current_decay * self.I + self.W_rec @ self.R + self.W_in @ (cfg.input_gain * u)
            self.I = np.clip(self.I, *cfg.i_clip).astype(np.float32)
            self._rk4_step()
        return self.R

    def features(self, input_vector):
        rate_features = self.R / max(1e-6, self.config.rate_feature_scale)
        return np.concatenate([rate_features, np.asarray(input_vector, dtype=np.float32), np.ones(1, dtype=np.float32)]).astype(np.float32)

    def frame(self):
        side = int(np.sqrt(self.config.n_reservoir))
        if side * side == self.config.n_reservoir:
            return self.R.reshape(side, side)
        return self.R[None, :]

    def stats(self):
        def summary(x):
            return {
                "mean": float(np.mean(x)),
                "std": float(np.std(x)),
                "min": float(np.min(x)),
                "max": float(np.max(x)),
            }
        return {
            "rate": summary(self.R),
            "voltage": summary(self.V),
            "current": summary(self.I),
            "Delta": summary(self.Delta),
            "Eta": summary(self.Eta),
            "current_decay": summary(self.current_decay),
        }


## Linear Actor-Critic Readout

The actor is a softmax linear policy over reservoir features. The critic is a linear value estimate over the same features. Updates use a compact PPO-style clipped surrogate objective with target-KL early stopping and Armijo backtracking on the fixed on-policy batch, so continued training has both a policy-ratio constraint and a sufficient-improvement check.

In [ ]:
class ReservoirActorCritic:
    def __init__(self, n_features: int, config: RLConfig):
        self.n_features = int(n_features)
        self.config = replace(config)
        self.rng = np.random.default_rng(config.seed + 2)
        self.actor_W = self.rng.normal(0.0, 1e-3, size=(self.n_features, 2)).astype(np.float32)
        self.value_W = np.zeros(self.n_features, dtype=np.float32)
        self.updates = 0
        self.last_value_loss = None
        self.last_policy_entropy = None
        self.last_episode_advantage_mean = None
        self.last_batch_episodes = 0
        self.last_batch_steps = 0
        self.last_actor_grad_norm = None
        self.last_value_grad_norm = None
        self.last_policy_logit_step = None
        self.last_policy_step_scale = 1.0
        self.last_approx_kl = None
        self.last_clip_fraction = None
        self.last_ppo_epochs = 0
        self.last_policy_loss = None
        self.last_line_search_steps = 0
        self.last_line_search_scale = 1.0
        self.last_line_search_accepted = True

    @property
    def n_trained_parameters(self):
        return int(self.actor_W.size + self.value_W.size)

    def reset_weights(self, seed: int | None = None):
        if seed is not None:
            self.rng = np.random.default_rng(int(seed))
        self.actor_W = self.rng.normal(0.0, 1e-3, size=self.actor_W.shape).astype(np.float32)
        self.value_W.fill(0.0)
        self.updates = 0
        self.last_value_loss = None
        self.last_policy_entropy = None
        self.last_episode_advantage_mean = None
        self.last_batch_episodes = 0
        self.last_batch_steps = 0
        self.last_actor_grad_norm = None
        self.last_value_grad_norm = None
        self.last_policy_logit_step = None
        self.last_policy_step_scale = 1.0
        self.last_approx_kl = None
        self.last_clip_fraction = None
        self.last_ppo_epochs = 0
        self.last_policy_loss = None
        self.last_line_search_steps = 0
        self.last_line_search_scale = 1.0
        self.last_line_search_accepted = True

    def logits(self, features):
        F = np.asarray(features, dtype=np.float32)
        if F.ndim == 1:
            F = F[None, :]
        return F @ self.actor_W

    def probabilities(self, features):
        return softmax(self.logits(features))

    def value(self, features):
        F = np.asarray(features, dtype=np.float32)
        return F @ self.value_W

    def choose_action(self, features, stochastic: bool = True):
        probs = self.probabilities(features)[0]
        if stochastic:
            action = int(self.rng.choice(2, p=probs))
        else:
            action = int(np.argmax(probs))
        return action, probs

    def _discounted_returns(self, rewards):
        rewards = np.asarray(rewards, dtype=np.float32)
        returns = np.zeros_like(rewards, dtype=np.float32)
        running_return = 0.0
        for idx in range(len(rewards) - 1, -1, -1):
            running_return = rewards[idx] + self.config.gamma * running_return
            returns[idx] = running_return
        return returns

    def _clip_gradient(self, grad):
        max_norm = float(getattr(self.config, "max_grad_norm", 0.0))
        norm = float(np.linalg.norm(grad))
        if not np.isfinite(norm):
            return np.zeros_like(grad, dtype=np.float32), norm
        if max_norm > 0.0 and norm > max_norm:
            grad = grad * (max_norm / (norm + 1e-8))
        return grad, norm

    def _policy_loss(self, actor_W, F, actions, old_log_probs, old_probs, actor_advantages, entropy_beta):
        logits = F @ actor_W
        logits = logits - logits.max(axis=1, keepdims=True)
        exp_logits = np.exp(logits)
        probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
        if not np.all(np.isfinite(probs)):
            return np.inf, np.nan, np.inf, np.inf, np.nan

        log_probs = np.log(probs[np.arange(len(actions)), actions] + 1e-8)
        ratio = np.exp(np.clip(log_probs - old_log_probs, -20.0, 20.0))
        ppo_clip = float(getattr(self.config, "ppo_clip", 0.0))
        if ppo_clip > 0.0:
            clipped_ratio = np.clip(ratio, 1.0 - ppo_clip, 1.0 + ppo_clip)
            surrogate = np.minimum(ratio * actor_advantages, clipped_ratio * actor_advantages)
            clip_fraction = float(np.mean(np.abs(ratio - 1.0) > ppo_clip))
        else:
            surrogate = ratio * actor_advantages
            clip_fraction = 0.0

        entropy = -np.sum(probs * np.log(probs + 1e-8), axis=1)
        entropy_mean = float(np.mean(entropy))
        old_log_full = np.log(old_probs + 1e-8)
        new_log_full = np.log(probs + 1e-8)
        full_kl = float(np.mean(np.sum(old_probs * (old_log_full - new_log_full), axis=1)))
        sampled_kl = float(np.mean(old_log_probs - log_probs))
        loss = -float(np.mean(surrogate)) - float(entropy_beta) * entropy_mean
        loss += 0.5 * float(self.config.l2) * float(np.sum(actor_W * actor_W))
        return loss, entropy_mean, max(0.0, full_kl), sampled_kl, clip_fraction

    def update_episodes(self, trajectories):
        """PPO-style actor-critic update with Armijo backtracking on a fixed batch.

        The batch is collected on-policy, then reused for the line search. This
        keeps the sufficient-improvement and KL checks deterministic for the
        tentative actor steps, which is the part classic line search needs.
        """
        feature_batches, action_batches, return_batches = [], [], []
        for trajectory in trajectories:
            if trajectory is None or len(trajectory.get("rewards", [])) == 0:
                continue
            feature_batches.append(np.asarray(trajectory["features"], dtype=np.float32))
            action_batches.append(np.asarray(trajectory["actions"], dtype=np.int64))
            return_batches.append(self._discounted_returns(trajectory["rewards"]))
        if not feature_batches:
            return None

        F = np.vstack(feature_batches).astype(np.float32)
        actions = np.concatenate(action_batches).astype(np.int64)
        returns = np.concatenate(return_batches).astype(np.float32)
        if not (np.all(np.isfinite(F)) and np.all(np.isfinite(returns))):
            return {"skipped": True, "reason": "non-finite features or returns"}

        old_probs = self.probabilities(F).astype(np.float32)
        if not np.all(np.isfinite(old_probs)):
            return {"skipped": True, "reason": "non-finite old policy probabilities"}
        old_log_probs = np.log(old_probs[np.arange(len(actions)), actions] + 1e-8).astype(np.float32)

        values_before = F @ self.value_W
        advantages = returns - values_before
        actor_advantages = advantages.copy()
        if self.config.normalize_advantage and len(actor_advantages) > 1:
            actor_advantages = (actor_advantages - actor_advantages.mean()) / (actor_advantages.std() + 1e-6)
        advantage_clip = float(getattr(self.config, "advantage_clip", 0.0))
        if advantage_clip > 0.0:
            actor_advantages = np.clip(actor_advantages, -advantage_clip, advantage_clip)

        selected = np.zeros((len(actions), 2), dtype=np.float32)
        selected[np.arange(len(actions)), actions] = 1.0

        ppo_epochs = max(1, int(getattr(self.config, "ppo_epochs", 1)))
        ppo_clip = float(getattr(self.config, "ppo_clip", 0.0))
        target_kl = float(getattr(self.config, "target_kl", 0.0))
        use_line_search = bool(getattr(self.config, "line_search", True))
        backtracks = max(1, int(getattr(self.config, "line_search_backtracks", 8)))
        decay = float(np.clip(getattr(self.config, "line_search_decay", 0.5), 0.05, 0.95))
        armijo_c1 = float(getattr(self.config, "armijo_c1", 1e-4))
        last_metrics = None

        for epoch in range(ppo_epochs):
            probs = self.probabilities(F)
            if not np.all(np.isfinite(probs)):
                return {"skipped": True, "reason": "non-finite policy probabilities"}
            log_probs = np.log(probs[np.arange(len(actions)), actions] + 1e-8)
            ratio = np.exp(np.clip(log_probs - old_log_probs, -20.0, 20.0))

            if ppo_clip > 0.0:
                active = (
                    ((actor_advantages >= 0.0) & (ratio <= 1.0 + ppo_clip))
                    | ((actor_advantages < 0.0) & (ratio >= 1.0 - ppo_clip))
                ).astype(np.float32)
            else:
                active = np.ones_like(actor_advantages, dtype=np.float32)

            entropy = -np.sum(probs * np.log(probs + 1e-8), axis=1)
            entropy_mean = float(np.mean(entropy))
            entropy_beta = float(self.config.entropy_beta)
            entropy_floor = float(getattr(self.config, "entropy_floor", 0.0))
            if entropy_floor > 0.0 and entropy_mean < entropy_floor:
                recovery = float(getattr(self.config, "entropy_recovery_beta", 0.0))
                entropy_beta = max(entropy_beta, recovery * (entropy_floor - entropy_mean) / max(entropy_floor, 1e-6))

            policy_weights = actor_advantages * ratio * active
            grad_logits = (probs - selected) * policy_weights[:, None]
            if entropy_beta != 0.0:
                entropy_grad = probs * (np.log(probs + 1e-8) + entropy[:, None])
                grad_logits += entropy_beta * entropy_grad

            values = F @ self.value_W
            grad_actor = F.T @ grad_logits / len(actions)
            grad_actor += self.config.l2 * self.actor_W
            grad_value = F.T @ (values - returns) / len(actions)
            grad_value += self.config.l2 * self.value_W
            grad_actor, actor_grad_norm = self._clip_gradient(grad_actor)
            grad_value, value_grad_norm = self._clip_gradient(grad_value)
            if not (np.all(np.isfinite(grad_actor)) and np.all(np.isfinite(grad_value))):
                return {"skipped": True, "reason": "non-finite gradients"}

            current_loss, _, current_kl, _, _ = self._policy_loss(
                self.actor_W, F, actions, old_log_probs, old_probs, actor_advantages, entropy_beta
            )
            if target_kl > 0.0 and current_kl > target_kl:
                break

            actor_step = -self.config.actor_lr * grad_actor.astype(np.float32)
            logit_step = F @ actor_step
            logit_step_rms = float(np.sqrt(np.mean(logit_step * logit_step)))
            max_logit_step = float(getattr(self.config, "max_policy_logit_step", 0.0))
            logit_step_scale = 1.0
            if max_logit_step > 0.0 and logit_step_rms > max_logit_step:
                logit_step_scale = max_logit_step / (logit_step_rms + 1e-8)
                actor_step *= logit_step_scale

            grad_dot_step = float(np.sum(grad_actor * actor_step))
            accepted = not use_line_search
            line_scale = 1.0
            line_steps = 0
            candidate_loss = current_loss
            candidate_entropy = entropy_mean
            candidate_kl = current_kl
            candidate_sampled_kl = 0.0
            candidate_clip_fraction = 0.0

            if use_line_search:
                accepted = False
                if grad_dot_step < 0.0 and np.isfinite(current_loss):
                    for line_steps in range(1, backtracks + 1):
                        candidate_W = self.actor_W + line_scale * actor_step
                        candidate_loss, candidate_entropy, candidate_kl, candidate_sampled_kl, candidate_clip_fraction = self._policy_loss(
                            candidate_W, F, actions, old_log_probs, old_probs, actor_advantages, entropy_beta
                        )
                        sufficient_decrease = candidate_loss <= current_loss + armijo_c1 * line_scale * grad_dot_step
                        kl_ok = (target_kl <= 0.0) or (candidate_kl <= target_kl)
                        if np.isfinite(candidate_loss) and sufficient_decrease and kl_ok:
                            accepted = True
                            break
                        line_scale *= decay
                if not accepted:
                    line_scale = 0.0
                    actor_step = np.zeros_like(actor_step, dtype=np.float32)
                    candidate_loss, candidate_entropy, candidate_kl, candidate_sampled_kl, candidate_clip_fraction = self._policy_loss(
                        self.actor_W, F, actions, old_log_probs, old_probs, actor_advantages, entropy_beta
                    )
            else:
                candidate_W = self.actor_W + actor_step
                candidate_loss, candidate_entropy, candidate_kl, candidate_sampled_kl, candidate_clip_fraction = self._policy_loss(
                    candidate_W, F, actions, old_log_probs, old_probs, actor_advantages, entropy_beta
                )

            self.actor_W += line_scale * actor_step
            self.value_W -= self.config.critic_lr * grad_value.astype(np.float32)
            self.actor_W = np.clip(self.actor_W, -10.0, 10.0)
            self.value_W = np.clip(self.value_W, -100.0, 100.0)

            last_metrics = {
                "value_loss": float(np.mean((values - returns) ** 2)),
                "policy_loss": float(candidate_loss),
                "entropy": float(candidate_entropy),
                "advantage_mean": float(np.mean(advantages)),
                "actor_grad_norm": float(actor_grad_norm),
                "value_grad_norm": float(value_grad_norm),
                "policy_logit_step": logit_step_rms,
                "policy_step_scale": float(logit_step_scale * line_scale),
                "approx_kl": float(candidate_kl),
                "sampled_kl": float(candidate_sampled_kl),
                "clip_fraction": float(candidate_clip_fraction),
                "line_search_steps": int(line_steps),
                "line_search_scale": float(line_scale),
                "line_search_accepted": bool(accepted),
                "ppo_epochs": epoch + 1,
                "batch_episodes": len(feature_batches),
                "batch_steps": int(len(actions)),
            }
            if use_line_search and not accepted:
                break
            if target_kl > 0.0 and candidate_kl > target_kl:
                break

        if last_metrics is None:
            return {"skipped": True, "reason": "no policy epoch ran"}

        self.updates += 1
        self.last_value_loss = last_metrics["value_loss"]
        self.last_policy_loss = last_metrics["policy_loss"]
        self.last_policy_entropy = last_metrics["entropy"]
        self.last_episode_advantage_mean = last_metrics["advantage_mean"]
        self.last_batch_episodes = last_metrics["batch_episodes"]
        self.last_batch_steps = last_metrics["batch_steps"]
        self.last_actor_grad_norm = last_metrics["actor_grad_norm"]
        self.last_value_grad_norm = last_metrics["value_grad_norm"]
        self.last_policy_logit_step = last_metrics["policy_logit_step"]
        self.last_policy_step_scale = last_metrics["policy_step_scale"]
        self.last_approx_kl = last_metrics["approx_kl"]
        self.last_clip_fraction = last_metrics["clip_fraction"]
        self.last_line_search_steps = last_metrics["line_search_steps"]
        self.last_line_search_scale = last_metrics["line_search_scale"]
        self.last_line_search_accepted = last_metrics["line_search_accepted"]
        self.last_ppo_epochs = last_metrics["ppo_epochs"]
        return last_metrics

    def update_episode(self, features, actions, rewards):
        return self.update_episodes([
            {"features": features, "actions": actions, "rewards": rewards}
        ])


## Episode And Training Loops

`run_episode` is shared by training, evaluation, and the live rollout view. Training stores the reservoir feature, action, reward, and old action log-probability at each decision. The actor-critic update consumes `batch_episodes` complete episodes and applies a PPO-style clipped policy update.

In [ ]:
def make_system(reservoir_config: ReservoirConfig = RESERVOIR_CONFIG, rl_config: RLConfig = RL_CONFIG):
    env = make_env(rl_config, seed=rl_config.seed)
    reservoir = Reservoir(reservoir_config)
    agent = ReservoirActorCritic(reservoir.n_features, rl_config)
    return env, reservoir, agent


def run_episode(
    env,
    reservoir,
    agent: ReservoirActorCritic,
    config: RLConfig,
    train: bool = True,
    stochastic: bool = True,
    seed: int | None = None,
    render_callback=None,
    delay_s: float = 0.0,
    update: bool = True,
    return_trajectory: bool = False,
):
    observation, _ = env.reset(seed=seed)
    reservoir.reset()
    previous_action = 0.0
    features, actions, rewards, log_probs = [], [], [], []
    total_reward = 0.0
    last_probs = np.full(2, 0.5, dtype=np.float32)
    terminated_flag = False
    truncated_flag = False

    for step in range(config.max_steps):
        input_vector = encode_cartpole_observation(
            env,
            observation,
            previous_action=previous_action,
            step_fraction=step / max(1, config.max_steps),
        )
        reservoir.step(input_vector)
        feature = reservoir.features(input_vector)
        action, probs = agent.choose_action(feature, stochastic=stochastic)
        last_probs = probs

        next_observation, reward, terminated, truncated, _ = env.step(action)
        terminated_flag = bool(terminated)
        truncated_flag = bool(truncated)
        done = bool(terminated_flag or truncated_flag)
        scaled_reward = float(reward) * config.reward_scale

        features.append(feature)
        actions.append(action)
        rewards.append(scaled_reward)
        log_probs.append(float(np.log(probs[action] + 1e-8)))
        total_reward += float(reward)

        if render_callback is not None:
            render_callback(
                observation=next_observation,
                action=action,
                probs=probs,
                step=step + 1,
                total_reward=total_reward,
                done=done,
            )
            if delay_s > 0.0:
                time.sleep(delay_s)

        previous_action = 1.0 if action == 1 else -1.0
        observation = next_observation
        if done:
            break

    trajectory = {"features": features, "actions": actions, "rewards": rewards, "log_probs": log_probs}
    metrics = None
    if train and update:
        metrics = agent.update_episodes([trajectory])
    result = {
        "return": float(total_reward),
        "steps": len(rewards),
        "terminated": terminated_flag,
        "truncated": truncated_flag,
        "last_probs": last_probs,
        "metrics": metrics,
    }
    if return_trajectory:
        result["trajectory"] = trajectory
    return result


def evaluate_agent(env, reservoir, agent, config: RLConfig, episodes: int | None = None, seed_offset: int = 10_000):
    episodes = config.eval_episodes if episodes is None else int(episodes)
    returns = []
    for idx in range(episodes):
        result = run_episode(
            env,
            reservoir,
            agent,
            config,
            train=False,
            stochastic=False,
            seed=seed_offset + idx,
            update=False,
        )
        returns.append(result["return"])
    return np.asarray(returns, dtype=np.float32)


def train_agent(
    env,
    reservoir,
    agent,
    config: RLConfig,
    episodes: int | None = None,
    eval_every: int | None = None,
    progress: bool = True,
):
    episodes = config.train_episodes if episodes is None else int(episodes)
    eval_every = config.eval_every if eval_every is None else int(eval_every)
    agent.config = replace(config)
    batch_episodes = max(1, int(getattr(config, "batch_episodes", 1)))
    returns = []
    eval_history = []
    pending_trajectories = []
    iterator = range(episodes)
    if progress:
        iterator = tqdm(iterator, total=episodes, desc="actor-critic episodes")
    for episode in iterator:
        result = run_episode(
            env,
            reservoir,
            agent,
            config,
            train=True,
            stochastic=True,
            seed=config.seed * 100_000 + episode,
            update=False,
            return_trajectory=True,
        )
        returns.append(result["return"])
        pending_trajectories.append(result["trajectory"])

        if len(pending_trajectories) >= batch_episodes or episode == episodes - 1:
            agent.update_episodes(pending_trajectories)
            pending_trajectories = []

        if eval_every and (episode + 1) % eval_every == 0:
            eval_returns = evaluate_agent(
                env,
                reservoir,
                agent,
                config,
                episodes=max(10, min(30, config.eval_episodes)),
                seed_offset=20_000 + episode * 100,
            )
            eval_history.append((episode + 1, float(eval_returns.mean()), float(eval_returns.min()), float(eval_returns.max())))
            if progress:
                iterator.set_postfix(
                    train_last50=f"{np.mean(returns[-50:]):.1f}",
                    eval=f"{eval_returns.mean():.1f}",
                    updates=agent.updates,
                )
    return {
        "returns": np.asarray(returns, dtype=np.float32),
        "eval_history": eval_history,
    }


## Build And Train

The MPR defaults use larger on-policy batches, gradient clipping, PPO-style policy-ratio clipping, and batch-local Armijo backtracking. This is aimed at continued interactive training: high-batch updates reduce return swings, while target-KL and line search prevent a good controller from being pushed too far by one noisy batch.

In [ ]:
env, reservoir, agent = make_system(RESERVOIR_CONFIG, RL_CONFIG)
print(f"reservoir neurons: {RESERVOIR_CONFIG.n_reservoir}")
print(f"feature dimension: {reservoir.n_features}")
print(f"trained parameters: {agent.n_trained_parameters}")
print("reservoir stats:", reservoir.stats())

start_time = time.perf_counter()
history = train_agent(env, reservoir, agent, RL_CONFIG)
train_seconds = time.perf_counter() - start_time

eval_returns = evaluate_agent(env, reservoir, agent, RL_CONFIG)
returns = history["returns"]
print(f"training time: {train_seconds:.2f}s")
print(f"train last 50: {returns[-50:].mean():.1f}")
print(f"train best 100: {max(np.mean(returns[max(0, idx - 100):idx]) for idx in range(100, len(returns) + 1)):.1f}")
print(f"greedy eval mean/min/max over {len(eval_returns)} episodes: {eval_returns.mean():.1f} / {eval_returns.min():.1f} / {eval_returns.max():.1f}")
print(f"last value loss: {agent.last_value_loss:.6f}; policy entropy: {agent.last_policy_entropy:.4f}")
print("final reservoir stats:", reservoir.stats())


## Training Curves

The policy is stochastic during training, so the raw returns are noisy. The greedy evaluation checkpoints are a better indicator of whether the readout has learned a useful controller.


In [ ]:
def plot_training_history(history, eval_returns=None):
    returns = np.asarray(history["returns"], dtype=np.float32)
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=returns, mode="lines", name="train return", line=dict(width=1, color="#9ecae1")))
    fig.add_trace(go.Scatter(y=moving_average(returns, 50), mode="lines", name="train MA50", line=dict(width=3, color="#1f77b4")))
    if history["eval_history"]:
        episodes, means, mins, maxs = zip(*history["eval_history"])
        fig.add_trace(go.Scatter(x=episodes, y=means, mode="lines+markers", name="greedy eval mean", line=dict(width=3, color="#f58518")))
        fig.add_trace(go.Scatter(x=episodes, y=maxs, mode="markers", name="greedy eval max", marker=dict(color="#54a24b", size=7)))
    if eval_returns is not None:
        fig.add_hline(y=float(np.mean(eval_returns)), line_dash="dot", line_color="#e45756", annotation_text="final eval mean")
    fig.update_layout(
        title="CartPole reservoir actor-critic training",
        xaxis_title="episode",
        yaxis_title="return before failure or 500-step cap",
        width=950,
        height=430,
        margin=dict(l=50, r=20, t=60, b=45),
    )
    fig.show()
    return fig

training_fig = plot_training_history(history, eval_returns)


## CartPole Drawing Helpers

The live view draws CartPole directly from the environment state, so it works without pygame or Gym render dependencies.


In [ ]:
def cartpole_traces(observation, env):
    x, _, theta, _ = [float(v) for v in observation]
    pole_length = float(env.unwrapped.length) * 2.0
    cart_half_width = 0.18
    cart_half_height = 0.08
    pivot_y = cart_half_height
    tip_x = x + pole_length * np.sin(theta)
    tip_y = pivot_y + pole_length * np.cos(theta)
    cart_x = [x - cart_half_width, x + cart_half_width, x + cart_half_width, x - cart_half_width, x - cart_half_width]
    cart_y = [-cart_half_height, -cart_half_height, cart_half_height, cart_half_height, -cart_half_height]
    return cart_x, cart_y, [x, tip_x], [pivot_y, tip_y]


def make_cartpole_figure(env):
    observation, _ = env.reset(seed=12345)
    cart_x, cart_y, pole_x, pole_y = cartpole_traces(observation, env)
    xlim = float(env.unwrapped.x_threshold) + 0.4
    fig = go.FigureWidget()
    fig.add_trace(go.Scatter(x=[-xlim, xlim], y=[-0.09, -0.09], mode="lines", name="track", line=dict(color="#555", width=2), showlegend=False))
    fig.add_trace(go.Scatter(x=cart_x, y=cart_y, mode="lines", fill="toself", name="cart", line=dict(color="#1f77b4", width=2), fillcolor="rgba(31,119,180,0.35)", showlegend=False))
    fig.add_trace(go.Scatter(x=pole_x, y=pole_y, mode="lines+markers", name="pole", line=dict(color="#f58518", width=5), marker=dict(size=[8, 10]), showlegend=False))
    fig.update_layout(
        width=620,
        height=330,
        margin=dict(l=20, r=20, t=32, b=25),
        title=dict(text="CartPole rollout", x=0.5, font=dict(size=14)),
        xaxis=dict(range=[-xlim, xlim], zeroline=False, fixedrange=True),
        yaxis=dict(range=[-0.25, 1.15], scaleanchor="x", scaleratio=1, zeroline=False, fixedrange=True),
    )
    return fig


def update_cartpole_figure(fig, observation, env):
    cart_x, cart_y, pole_x, pole_y = cartpole_traces(observation, env)
    with fig.batch_update():
        fig.data[1].x = cart_x
        fig.data[1].y = cart_y
        fig.data[2].x = pole_x
        fig.data[2].y = pole_y


## Live RL Workbench

Use this after the training cell. You can keep training the same actor-critic readout, evaluate it, or watch a greedy rollout. Reservoir weights are fixed; reset agent only clears the trainable actor and critic readouts.


In [ ]:
class CartPoleRLWorkbench:
    def __init__(self, env, reservoir, agent: ReservoirActorCritic, config: RLConfig, history=None):
        self.env = env
        self.reservoir = reservoir
        self.agent = agent
        self.config = replace(config)
        self.history = {"returns": [], "eval_history": []} if history is None else {
            "returns": list(np.asarray(history["returns"], dtype=float)),
            "eval_history": list(history.get("eval_history", [])),
        }
        self._stop = threading.Event()
        self._thread = None
        self._lock = threading.RLock()

        self.world_fig = make_cartpole_figure(env)
        self.policy_fig = go.FigureWidget(data=[go.Bar(x=["left", "right"], y=[0.5, 0.5], marker_color=["#4c78a8", "#4c78a8"])])
        self.policy_fig.update_layout(width=310, height=260, yaxis=dict(range=[0, 1], fixedrange=True), margin=dict(l=35, r=10, t=32, b=35), title=dict(text="policy probabilities", x=0.5, font=dict(size=14)))
        self.reservoir_fig = go.FigureWidget(data=[go.Heatmap(z=self.reservoir.frame(), colorscale="RdBu", zmin=-1, zmax=1, showscale=False)])
        self.reservoir_fig.update_layout(width=310, height=310, margin=dict(l=0, r=0, t=32, b=0), title=dict(text=self.reservoir.frame_title, x=0.5, font=dict(size=14)), xaxis=dict(visible=False), yaxis=dict(visible=False, autorange="reversed"))
        self.return_fig = go.FigureWidget()
        self.return_fig.update_layout(width=950, height=320, margin=dict(l=45, r=20, t=35, b=40), xaxis_title="episode", yaxis_title="return", title=dict(text="live training returns", x=0.5, font=dict(size=14)))
        self._refresh_return_figure()

        self.train_button = widgets.Button(description="Train", icon="graduation-cap", button_style="success")
        self.eval_button = widgets.Button(description="Eval", icon="check", button_style="info")
        self.rollout_button = widgets.Button(description="Rollout", icon="play", button_style="primary")
        self.stop_button = widgets.Button(description="Stop", icon="stop", button_style="danger")
        self.reset_agent_button = widgets.Button(description="Reset Agent", icon="eraser")

        self.train_episodes = widgets.IntSlider(value=100, min=1, max=3000, step=1, description="episodes", continuous_update=False)
        self.eval_episodes = widgets.IntSlider(value=20, min=1, max=100, step=1, description="eval n", continuous_update=False)
        self.batch_episodes = widgets.IntSlider(value=int(getattr(self.agent.config, "batch_episodes", 1)), min=1, max=128, step=1, description="batch eps", continuous_update=False, style={"description_width": "initial"})
        self.delay_ms = widgets.IntSlider(value=20, min=0, max=200, step=5, description="delay ms", continuous_update=False)
        self.stochastic_rollout = widgets.Checkbox(value=False, description="stochastic rollout")
        self.actor_lr = widgets.FloatLogSlider(value=self.agent.config.actor_lr, base=10, min=-4, max=0, step=0.1, description="actor lr", continuous_update=False, style={"description_width": "initial"})
        self.critic_lr = widgets.FloatLogSlider(value=self.agent.config.critic_lr, base=10, min=-5, max=0, step=0.1, description="critic lr", continuous_update=False, style={"description_width": "initial"})
        self.gamma = widgets.FloatSlider(value=self.agent.config.gamma, min=0.90, max=0.999, step=0.001, readout_format=".3f", description="gamma", continuous_update=False)
        self.reward_scale = widgets.FloatLogSlider(value=self.agent.config.reward_scale, base=10, min=-4, max=0, step=0.1, description="reward scale", continuous_update=False, style={"description_width": "initial"})
        self.policy_step = widgets.FloatSlider(value=float(getattr(self.agent.config, "max_policy_logit_step", 0.08)), min=0.01, max=0.25, step=0.01, readout_format=".2f", description="policy step", continuous_update=False, style={"description_width": "initial"})
        self.entropy_floor = widgets.FloatSlider(value=float(getattr(self.agent.config, "entropy_floor", 0.45)), min=0.0, max=0.68, step=0.01, readout_format=".2f", description="entropy min", continuous_update=False, style={"description_width": "initial"})
        self.ppo_clip = widgets.FloatSlider(value=float(getattr(self.agent.config, "ppo_clip", 0.20)), min=0.05, max=0.40, step=0.01, readout_format=".2f", description="ppo clip", continuous_update=False, style={"description_width": "initial"})
        self.target_kl = widgets.FloatSlider(value=float(getattr(self.agent.config, "target_kl", 0.02)), min=0.0, max=0.10, step=0.005, readout_format=".3f", description="target kl", continuous_update=False, style={"description_width": "initial"})
        self.ppo_epochs = widgets.IntSlider(value=int(getattr(self.agent.config, "ppo_epochs", 4)), min=1, max=8, step=1, description="ppo epochs", continuous_update=False, style={"description_width": "initial"})
        self.line_search = widgets.Checkbox(value=bool(getattr(self.agent.config, "line_search", True)), description="line search")
        self.backtracks = widgets.IntSlider(value=int(getattr(self.agent.config, "line_search_backtracks", 8)), min=1, max=12, step=1, description="backtracks", continuous_update=False, style={"description_width": "initial"})
        self.status = widgets.HTML(value="idle")

        self.train_button.on_click(lambda _: self.train_async())
        self.eval_button.on_click(lambda _: self.evaluate_async())
        self.rollout_button.on_click(lambda _: self.rollout_async())
        self.stop_button.on_click(lambda _: self.stop())
        self.reset_agent_button.on_click(lambda _: self.reset_agent())

    def _sync_config(self):
        self.agent.config.actor_lr = float(self.actor_lr.value)
        self.agent.config.critic_lr = float(self.critic_lr.value)
        self.agent.config.gamma = float(self.gamma.value)
        self.agent.config.reward_scale = float(self.reward_scale.value)
        self.agent.config.batch_episodes = int(self.batch_episodes.value)
        self.agent.config.max_policy_logit_step = float(self.policy_step.value)
        self.agent.config.entropy_floor = float(self.entropy_floor.value)
        self.agent.config.ppo_clip = float(self.ppo_clip.value)
        self.agent.config.target_kl = float(self.target_kl.value)
        self.agent.config.ppo_epochs = int(self.ppo_epochs.value)
        self.agent.config.line_search = bool(self.line_search.value)
        self.agent.config.line_search_backtracks = int(self.backtracks.value)
        self.config = replace(
            self.config,
            actor_lr=float(self.actor_lr.value),
            critic_lr=float(self.critic_lr.value),
            gamma=float(self.gamma.value),
            reward_scale=float(self.reward_scale.value),
            batch_episodes=int(self.batch_episodes.value),
            max_policy_logit_step=float(self.policy_step.value),
            entropy_floor=float(self.entropy_floor.value),
            ppo_clip=float(self.ppo_clip.value),
            target_kl=float(self.target_kl.value),
            ppo_epochs=int(self.ppo_epochs.value),
            line_search=bool(self.line_search.value),
            line_search_backtracks=int(self.backtracks.value),
        )

    def _refresh_return_figure(self):
        returns = np.asarray(self.history["returns"], dtype=np.float32)
        with self.return_fig.batch_update():
            self.return_fig.data = []
            if len(returns):
                self.return_fig.add_trace(go.Scatter(y=returns, mode="lines", name="return", line=dict(color="#9ecae1", width=1)))
                self.return_fig.add_trace(go.Scatter(y=moving_average(returns, 50), mode="lines", name="MA50", line=dict(color="#1f77b4", width=3)))
            if self.history["eval_history"]:
                episodes, means, mins, maxs = zip(*self.history["eval_history"])
                self.return_fig.add_trace(go.Scatter(x=episodes, y=means, mode="lines+markers", name="eval mean", line=dict(color="#f58518", width=3)))

    def _draw_policy(self, probs):
        pred = int(np.argmax(probs))
        colors = ["#4c78a8", "#4c78a8"]
        colors[pred] = "#f58518"
        with self.policy_fig.batch_update():
            self.policy_fig.data[0].y = probs
            self.policy_fig.data[0].marker.color = colors
        with self.reservoir_fig.batch_update():
            self.reservoir_fig.data[0].z = self.reservoir.frame()

    def _render_callback(self, observation, action, probs, step, total_reward, done):
        with self._lock:
            update_cartpole_figure(self.world_fig, observation, self.env)
            self._draw_policy(probs)
            self.status.value = f"rollout step={step} return={total_reward:.0f} action={action} done={done}"

    def train(self, episodes: int | None = None):
        self._sync_config()
        episodes = int(self.train_episodes.value if episodes is None else episodes)
        start_count = len(self.history["returns"])
        pending_trajectories = []
        batch_episodes = max(1, int(self.batch_episodes.value))
        for idx in range(episodes):
            if self._stop.is_set():
                break
            result = run_episode(
                self.env,
                self.reservoir,
                self.agent,
                self.agent.config,
                train=True,
                stochastic=True,
                seed=self.config.seed * 100_000 + start_count + idx,
                update=False,
                return_trajectory=True,
            )
            self.history["returns"].append(result["return"])
            pending_trajectories.append(result["trajectory"])
            if len(pending_trajectories) >= batch_episodes or idx == episodes - 1:
                self.agent.update_episodes(pending_trajectories)
                pending_trajectories = []
            if (idx + 1) % 10 == 0 or idx == episodes - 1:
                returns = np.asarray(self.history["returns"], dtype=np.float32)
                ma50 = float(np.mean(returns[-50:])) if len(returns) else np.nan
                with self._lock:
                    self._refresh_return_figure()
                    grad = self.agent.last_actor_grad_norm
                    step = self.agent.last_policy_logit_step
                    scale = self.agent.last_policy_step_scale
                    grad_text = "n/a" if grad is None else f"{grad:.2g}"
                    step_text = "n/a" if step is None else f"{step:.2g}"
                    kl = self.agent.last_approx_kl
                    kl_text = "n/a" if kl is None else f"{kl:.3f}"
                    bt = self.agent.last_line_search_steps
                    ls = self.agent.last_line_search_scale
                    self.status.value = f"trained={len(returns)} last={result['return']:.0f} MA50={ma50:.1f} updates={self.agent.updates} grad={grad_text} step={step_text} kl={kl_text} scale={scale:.2f} bt={bt} ls={ls:.2f}"
        self._stop.clear()

    def evaluate(self):
        self._sync_config()
        with self._lock:
            self.status.value = "evaluating..."
        returns = evaluate_agent(
            self.env,
            self.reservoir,
            self.agent,
            self.agent.config,
            episodes=int(self.eval_episodes.value),
            seed_offset=50_000 + len(self.history["returns"]) * 100,
        )
        self.history["eval_history"].append((len(self.history["returns"]), float(returns.mean()), float(returns.min()), float(returns.max())))
        with self._lock:
            self._refresh_return_figure()
            self.status.value = f"eval mean/min/max={returns.mean():.1f}/{returns.min():.0f}/{returns.max():.0f}"

    def rollout(self):
        self._sync_config()
        self._stop.clear()
        result = run_episode(
            self.env,
            self.reservoir,
            self.agent,
            self.agent.config,
            train=False,
            stochastic=bool(self.stochastic_rollout.value),
            seed=80_000 + len(self.history["returns"]),
            render_callback=self._render_callback,
            delay_s=float(self.delay_ms.value) / 1000.0,
        )
        with self._lock:
            self.status.value = f"rollout return={result['return']:.0f} steps={result['steps']}"

    def _start_thread(self, target):
        self.stop()
        self._stop.clear()
        self._thread = threading.Thread(target=target, daemon=True)
        self._thread.start()

    def train_async(self):
        self._start_thread(self.train)

    def evaluate_async(self):
        self._start_thread(self.evaluate)

    def rollout_async(self):
        self._start_thread(self.rollout)

    def stop(self):
        self._stop.set()
        if self._thread is not None and self._thread.is_alive():
            self._thread.join(timeout=2)

    def reset_agent(self):
        self.stop()
        with self._lock:
            self.agent.reset_weights(seed=self.config.seed + self.agent.updates + 10)
            self.history = {"returns": [], "eval_history": []}
            self._refresh_return_figure()
            self.status.value = "agent reset"

    def display(self):
        controls = widgets.VBox([
            widgets.HBox([self.train_button, self.eval_button, self.rollout_button, self.stop_button, self.reset_agent_button]),
            widgets.HBox([self.train_episodes, self.eval_episodes, self.batch_episodes, self.delay_ms, self.stochastic_rollout]),
            widgets.HBox([self.actor_lr, self.critic_lr, self.gamma, self.reward_scale]),
            widgets.HBox([self.policy_step, self.entropy_floor, self.ppo_clip, self.target_kl, self.ppo_epochs, self.line_search, self.backtracks]),
            self.status,
        ])
        display(widgets.VBox([
            controls,
            widgets.HBox([self.world_fig, widgets.VBox([self.policy_fig, self.reservoir_fig])]),
            self.return_fig,
        ]))


try:
    cartpole_workbench.stop()
except NameError:
    pass

cartpole_workbench = CartPoleRLWorkbench(env, reservoir, agent, RL_CONFIG, history)
cartpole_workbench.display()


## MPR Cell-Parameter Distribution Optimization

This section uses **Optuna/TPE** instead of random search. TPE is a bounded, derivative-free Bayesian optimizer that is a better fit than Nelder-Mead for this objective: each evaluation includes noisy RL training, the score is non-smooth, and parameters have hard bounds.

The optimizer searches both per-cell `Delta`, `Eta`, and `J` distribution parameters and actor-critic training hyperparameters, including PPO clip, target-KL, Armijo line-search, policy-step, and entropy-collapse controls. The scalar objective rewards high greedy-eval mean, penalizes high eval variance, and gives a small bonus for high stochastic training return, so it is less likely to select fragile one-off policies. The baseline config above is left unchanged; use the best optimized result to rebuild `RESERVOIR_CONFIG` for a longer confirmation run.

In [ ]:
import warnings
import optuna

warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

MPR_PARAMETER_BOUNDS = {
    "Delta": (0.0, 5.0),
    "Delta_std": (0.0, 3.0),
    "Eta": (-9.0, -1.0),
    "Eta_std": (0.0, 3.0),
    "J": (5.0, 15.0),
    "J_std": (0.0, 3.0),
}

TRAINING_PARAMETER_BOUNDS = {
    "gamma": (0.90, 0.98),
    "actor_lr": (1e-2, 1.0),
    "critic_lr": (1e-3, 8e-2),
    "reward_scale": (5e-3, 5e-2),
    "entropy_beta": (0.0, 2e-4),
    "max_grad_norm": (1.0, 10.0),
    "batch_episodes": (4, 8, 16, 32, 64),
    "max_policy_logit_step": (0.03, 0.20),
    "entropy_floor": (0.20, 0.60),
    "ppo_clip": (0.10, 0.35),
    "ppo_epochs": (1, 2, 4, 6, 8),
    "target_kl": (0.005, 0.08),
    "line_search_backtracks": (4, 6, 8, 10),
}

# Backward-compatible name if you used the previous random-search cell.
MPR_PARAMETER_SEARCH_SPACE = MPR_PARAMETER_BOUNDS


def mpr_config_from_params(params, base_config: ReservoirConfig = RESERVOIR_CONFIG, seed: int | None = None):
    """Build a ReservoirConfig from optimizer parameters."""
    cfg_seed = base_config.seed if seed is None else int(seed)
    return replace(
        base_config,
        seed=cfg_seed,
        Delta=float(params.get("Delta", base_config.Delta)),
        Delta_std=float(params.get("Delta_std", base_config.Delta_std)),
        Delta_clip=MPR_PARAMETER_BOUNDS["Delta"],
        Eta=float(params.get("Eta", base_config.Eta)),
        Eta_std=float(params.get("Eta_std", base_config.Eta_std)),
        Eta_clip=MPR_PARAMETER_BOUNDS["Eta"],
        J=float(params.get("J", base_config.J)),
        J_std=float(params.get("J_std", base_config.J_std)),
        J_clip=MPR_PARAMETER_BOUNDS["J"],
    )


def rl_config_from_params(params, base_config: RLConfig = RL_CONFIG, seed: int | None = None):
    """Build an RLConfig from optimizer parameters."""
    cfg_seed = base_config.seed if seed is None else int(seed)
    return replace(
        base_config,
        seed=cfg_seed,
        gamma=float(params.get("gamma", base_config.gamma)),
        actor_lr=float(params.get("actor_lr", base_config.actor_lr)),
        critic_lr=float(params.get("critic_lr", base_config.critic_lr)),
        reward_scale=float(params.get("reward_scale", base_config.reward_scale)),
        entropy_beta=float(params.get("entropy_beta", base_config.entropy_beta)),
        max_grad_norm=float(params.get("max_grad_norm", base_config.max_grad_norm)),
        batch_episodes=int(params.get("batch_episodes", base_config.batch_episodes)),
        max_policy_logit_step=float(params.get("max_policy_logit_step", base_config.max_policy_logit_step)),
        entropy_floor=float(params.get("entropy_floor", base_config.entropy_floor)),
        ppo_clip=float(params.get("ppo_clip", base_config.ppo_clip)),
        ppo_epochs=int(params.get("ppo_epochs", base_config.ppo_epochs)),
        target_kl=float(params.get("target_kl", base_config.target_kl)),
        line_search_backtracks=int(params.get("line_search_backtracks", base_config.line_search_backtracks)),
    )


def sample_mpr_parameter_config(rng, base_config: ReservoirConfig = RESERVOIR_CONFIG, trial_seed: int | None = None):
    """Uniform sample from the same bounds; useful for baseline comparisons."""
    params = {
        name: float(rng.uniform(low, high))
        for name, (low, high) in MPR_PARAMETER_BOUNDS.items()
    }
    seed = int(rng.integers(0, 2**31 - 1) if trial_seed is None else trial_seed)
    return mpr_config_from_params(params, base_config=base_config, seed=seed)


def suggest_mpr_parameters(trial):
    """Optuna search space for the MPR cell-parameter distribution."""
    return {
        name: trial.suggest_float(name, low, high)
        for name, (low, high) in MPR_PARAMETER_BOUNDS.items()
    }


def suggest_training_parameters(trial):
    """Optuna search space for the actor-critic training process."""
    return {
        "gamma": trial.suggest_float("gamma", *TRAINING_PARAMETER_BOUNDS["gamma"]),
        "actor_lr": trial.suggest_float("actor_lr", *TRAINING_PARAMETER_BOUNDS["actor_lr"], log=True),
        "critic_lr": trial.suggest_float("critic_lr", *TRAINING_PARAMETER_BOUNDS["critic_lr"], log=True),
        "reward_scale": trial.suggest_float("reward_scale", *TRAINING_PARAMETER_BOUNDS["reward_scale"], log=True),
        "entropy_beta": trial.suggest_float("entropy_beta", *TRAINING_PARAMETER_BOUNDS["entropy_beta"]),
        "max_grad_norm": trial.suggest_float("max_grad_norm", *TRAINING_PARAMETER_BOUNDS["max_grad_norm"], log=True),
        "batch_episodes": trial.suggest_categorical("batch_episodes", list(TRAINING_PARAMETER_BOUNDS["batch_episodes"])),
        "max_policy_logit_step": trial.suggest_float("max_policy_logit_step", *TRAINING_PARAMETER_BOUNDS["max_policy_logit_step"]),
        "entropy_floor": trial.suggest_float("entropy_floor", *TRAINING_PARAMETER_BOUNDS["entropy_floor"]),
        "ppo_clip": trial.suggest_float("ppo_clip", *TRAINING_PARAMETER_BOUNDS["ppo_clip"]),
        "ppo_epochs": trial.suggest_categorical("ppo_epochs", list(TRAINING_PARAMETER_BOUNDS["ppo_epochs"])),
        "target_kl": trial.suggest_float("target_kl", *TRAINING_PARAMETER_BOUNDS["target_kl"]),
        "line_search_backtracks": trial.suggest_categorical("line_search_backtracks", list(TRAINING_PARAMETER_BOUNDS["line_search_backtracks"])),
    }


def evaluate_mpr_candidate(
    reservoir_config: ReservoirConfig,
    rl_config: RLConfig,
    train_episodes: int,
    eval_episodes: int,
    eval_interval: int | None = None,
    trial=None,
):
    """Train one candidate and return a robust scalar optimization score.

    The score rewards high greedy-eval mean, penalizes eval variance, and gives
    a small bonus for high stochastic training return. This avoids selecting
    fragile policies that only look good on a few deterministic eval seeds.
    """
    trial_env, trial_reservoir, trial_agent = make_system(reservoir_config, rl_config)
    returns = []
    pending_trajectories = []
    batch_episodes = max(1, int(getattr(rl_config, "batch_episodes", 1)))
    train_episodes = int(train_episodes)
    eval_episodes = int(eval_episodes)
    if eval_interval is not None:
        eval_interval = int(max(1, eval_interval))

    for episode in range(train_episodes):
        result = run_episode(
            trial_env,
            trial_reservoir,
            trial_agent,
            rl_config,
            train=True,
            stochastic=True,
            seed=rl_config.seed * 100_000 + episode,
            update=False,
            return_trajectory=True,
        )
        returns.append(result["return"])
        pending_trajectories.append(result["trajectory"])
        if len(pending_trajectories) >= batch_episodes or episode == train_episodes - 1:
            trial_agent.update_episodes(pending_trajectories)
            pending_trajectories = []

        should_checkpoint = (
            trial is not None
            and eval_interval is not None
            and (episode + 1) % eval_interval == 0
            and episode + 1 < train_episodes
        )
        if should_checkpoint:
            checkpoint_eval = evaluate_agent(
                trial_env,
                trial_reservoir,
                trial_agent,
                rl_config,
                episodes=max(5, min(10, eval_episodes)),
                seed_offset=70_000 + trial.number * 1_000 + episode,
            )
            trial.report(float(checkpoint_eval.mean()), step=episode + 1)
            if trial.should_prune():
                raise optuna.TrialPruned()

    eval_returns = evaluate_agent(
        trial_env,
        trial_reservoir,
        trial_agent,
        rl_config,
        episodes=eval_episodes,
        seed_offset=90_000 + rl_config.seed,
    )
    returns = np.asarray(returns, dtype=np.float32)
    eval_mean = float(eval_returns.mean())
    eval_std = float(eval_returns.std())
    train_last50 = float(returns[-50:].mean()) if len(returns) else np.nan
    robust_score = eval_mean - 0.25 * eval_std + 0.10 * train_last50
    return {
        "score": float(robust_score),
        "eval_mean": eval_mean,
        "eval_std": eval_std,
        "eval_min": float(eval_returns.min()),
        "eval_max": float(eval_returns.max()),
        "train_last50": train_last50,
        "updates": int(trial_agent.updates),
        "history": {"returns": returns, "eval_history": []},
        "eval_returns": eval_returns,
        "reservoir_stats": trial_reservoir.stats(),
    }


def run_mpr_optuna_search(
    n_trials: int = 50,
    train_episodes: int = 400,
    eval_episodes: int = 20,
    eval_interval: int | None = 100,
    base_reservoir_config: ReservoirConfig = RESERVOIR_CONFIG,
    base_rl_config: RLConfig = RL_CONFIG,
    seed: int = 123,
    optimize_training: bool = True,
    study_name: str | None = None,
    storage: str | None = None,
    load_if_exists: bool = True,
    progress: bool = True,
):
    """Optimize MPR cell distributions and, by default, training hyperparameters.

    TPE uses previous completed trials to bias future samples toward promising
    bounded regions. Median pruning can terminate poor candidates at intermediate
    eval checkpoints, saving time during larger sweeps.
    """
    startup = max(5, min(12, int(n_trials) // 4 if int(n_trials) >= 8 else int(n_trials)))
    sampler = optuna.samplers.TPESampler(
        seed=int(seed),
        n_startup_trials=startup,
        multivariate=True,
        group=True,
    )
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=max(3, min(startup, int(n_trials))),
        n_warmup_steps=1,
    )
    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        study_name=study_name,
        storage=storage,
        load_if_exists=load_if_exists,
    )

    pbar = tqdm(total=int(n_trials), desc="Optuna MPR TPE", disable=not progress)

    def objective(trial):
        params = suggest_mpr_parameters(trial)
        if optimize_training:
            params.update(suggest_training_parameters(trial))
        trial_seed = int(seed) * 10_000 + trial.number
        reservoir_config = mpr_config_from_params(
            params,
            base_config=base_reservoir_config,
            seed=trial_seed,
        )
        rl_config = rl_config_from_params(
            params,
            base_config=base_rl_config,
            seed=trial_seed,
        )
        rl_config = replace(
            rl_config,
            train_episodes=int(train_episodes),
            eval_episodes=int(eval_episodes),
            eval_every=0,
        )
        result = evaluate_mpr_candidate(
            reservoir_config,
            rl_config,
            train_episodes=int(train_episodes),
            eval_episodes=int(eval_episodes),
            eval_interval=eval_interval,
            trial=trial,
        )
        trial.set_user_attr("config_seed", trial_seed)
        trial.set_user_attr("train_episodes", int(train_episodes))
        trial.set_user_attr("eval_episodes", int(eval_episodes))
        trial.set_user_attr("optimize_training", bool(optimize_training))
        trial.set_user_attr("eval_mean", result["eval_mean"])
        trial.set_user_attr("eval_std", result["eval_std"])
        trial.set_user_attr("eval_min", result["eval_min"])
        trial.set_user_attr("eval_max", result["eval_max"])
        trial.set_user_attr("train_last50", result["train_last50"])
        trial.set_user_attr("updates", result["updates"])
        trial.set_user_attr("reservoir_stats", result["reservoir_stats"])
        return result["score"]

    def update_progress(study, trial):
        pbar.update(1)
        try:
            pbar.set_postfix(best=f"{study.best_value:.1f}", last=trial.state.name)
        except ValueError:
            pbar.set_postfix(best="n/a", last=trial.state.name)

    study.optimize(
        objective,
        n_trials=int(n_trials),
        callbacks=[update_progress],
        show_progress_bar=False,
        gc_after_trial=True,
    )
    pbar.close()
    return study, optuna_mpr_results(study, base_reservoir_config, base_rl_config)


def optuna_mpr_results(
    study,
    base_reservoir_config: ReservoirConfig = RESERVOIR_CONFIG,
    base_rl_config: RLConfig = RL_CONFIG,
):
    """Convert completed Optuna trials into notebook-friendly result dicts."""
    results = []
    for trial in study.trials:
        if trial.state.name != "COMPLETE" or trial.value is None:
            continue
        trial_seed = int(trial.user_attrs.get("config_seed", base_reservoir_config.seed))
        train_episodes = int(trial.user_attrs.get("train_episodes", base_rl_config.train_episodes))
        eval_episodes = int(trial.user_attrs.get("eval_episodes", base_rl_config.eval_episodes))
        reservoir_config = mpr_config_from_params(
            trial.params,
            base_config=base_reservoir_config,
            seed=trial_seed,
        )
        rl_config = rl_config_from_params(
            trial.params,
            base_config=base_rl_config,
            seed=trial_seed,
        )
        rl_config = replace(
            rl_config,
            train_episodes=train_episodes,
            eval_episodes=eval_episodes,
            eval_every=0,
        )
        results.append({
            "trial": int(trial.number),
            "score": float(trial.value),
            "eval_mean": float(trial.user_attrs.get("eval_mean", trial.value)),
            "eval_std": float(trial.user_attrs.get("eval_std", np.nan)),
            "eval_min": float(trial.user_attrs.get("eval_min", np.nan)),
            "eval_max": float(trial.user_attrs.get("eval_max", np.nan)),
            "train_last50": float(trial.user_attrs.get("train_last50", np.nan)),
            "updates": int(trial.user_attrs.get("updates", 0)),
            "reservoir_config": reservoir_config,
            "rl_config": rl_config,
            "params": dict(trial.params),
            "reservoir_stats": trial.user_attrs.get("reservoir_stats", None),
            "trial_state": trial.state.name,
        })
    results.sort(key=lambda item: item["score"], reverse=True)
    return results


def summarize_search_result(result):
    rcfg = result["reservoir_config"]
    lcfg = result["rl_config"]
    return {
        "trial": result["trial"],
        "score": result["score"],
        "eval_mean": result.get("eval_mean", result["score"]),
        "eval_std": result.get("eval_std", np.nan),
        "eval_min": result["eval_min"],
        "eval_max": result["eval_max"],
        "train_last50": result["train_last50"],
        "updates": result.get("updates", np.nan),
        "Delta": rcfg.Delta,
        "Delta_std": rcfg.Delta_std,
        "Eta": rcfg.Eta,
        "Eta_std": rcfg.Eta_std,
        "J": rcfg.J,
        "J_std": rcfg.J_std,
        "gamma": lcfg.gamma,
        "actor_lr": lcfg.actor_lr,
        "critic_lr": lcfg.critic_lr,
        "reward_scale": lcfg.reward_scale,
        "entropy_beta": lcfg.entropy_beta,
        "batch_episodes": lcfg.batch_episodes,
        "max_grad_norm": lcfg.max_grad_norm,
        "max_policy_logit_step": lcfg.max_policy_logit_step,
        "entropy_floor": lcfg.entropy_floor,
        "ppo_clip": lcfg.ppo_clip,
        "ppo_epochs": lcfg.ppo_epochs,
        "target_kl": lcfg.target_kl,
        "line_search_backtracks": lcfg.line_search_backtracks,
        "seed": rcfg.seed,
    }


def print_mpr_optimizer_leaderboard(results, top_k: int = 10):
    if not results:
        print("no completed optimization results")
        return []
    rows = [summarize_search_result(result) for result in results[:top_k]]
    for row in rows:
        print(
            f"trial={row['trial']:03d} score={row['score']:.1f} eval_mean={row['eval_mean']:.1f} "
            f"eval={row['eval_min']:.0f}-{row['eval_max']:.0f} train50={row['train_last50']:.1f} updates={row['updates']} | "
            f"Delta={row['Delta']:.3f}+/-{row['Delta_std']:.3f} "
            f"Eta={row['Eta']:.3f}+/-{row['Eta_std']:.3f} "
            f"J={row['J']:.3f}+/-{row['J_std']:.3f} | "
            f"gamma={row['gamma']:.3f} actor_lr={row['actor_lr']:.2e} critic_lr={row['critic_lr']:.2e} "
            f"reward_scale={row['reward_scale']:.3g} batch={row['batch_episodes']} grad_clip={row['max_grad_norm']:.2f} "
            f"policy_step={row['max_policy_logit_step']:.2f} entropy_floor={row['entropy_floor']:.2f} "
            f"ppo_clip={row['ppo_clip']:.2f} ppo_epochs={row['ppo_epochs']} target_kl={row['target_kl']:.3f} "
            f"backtracks={row['line_search_backtracks']} seed={row['seed']}"
        )
    return rows


# Backward-compatible function name from the previous random-search section.
print_mpr_search_leaderboard = print_mpr_optimizer_leaderboard


def apply_optimized_result(result, train_episodes: int | None = None):
    """Promote an optimization result into notebook-level configs for a longer rerun."""
    global RESERVOIR_CONFIG, RL_CONFIG
    RESERVOIR_CONFIG = result["reservoir_config"]
    if train_episodes is None:
        RL_CONFIG = result["rl_config"]
    else:
        RL_CONFIG = replace(
            result["rl_config"],
            train_episodes=int(train_episodes),
            eval_every=RL_CONFIG.eval_every,
        )
    return RESERVOIR_CONFIG, RL_CONFIG


# Backward-compatible function name from the previous random-search section.
apply_search_result = apply_optimized_result


# Example usage. Start small, then increase n_trials and train_episodes for a real sweep.
study, optimization_results = run_mpr_optuna_search(
     n_trials=50,
     train_episodes=400,
     eval_episodes=20,
     eval_interval=100,
     seed=123,
     optimize_training=True,
 )
leaderboard = print_mpr_optimizer_leaderboard(optimization_results, top_k=10)
RESERVOIR_CONFIG, RL_CONFIG = apply_optimized_result(optimization_results[0], train_episodes=1500)
env, reservoir, agent = make_system(RESERVOIR_CONFIG, RL_CONFIG)


In [ ]:
# Optional: after running the Optuna cell above, promote the best result and do a longer confirmation run.

#
RESERVOIR_CONFIG, RL_CONFIG = apply_optimized_result(optimization_results[0], train_episodes=10000)
RL_CONFIG.batch_episodes = 32
RL_CONFIG.ppo_epochs = 4
RL_CONFIG.armijo_c1 = 0
RL_CONFIG.target_kl = 0.015
RL_CONFIG.entropy_floor = .4
RL_CONFIG.max_policy_logit_step = 0.08
RL_CONFIG.critic_lr = 0.05
env, reservoir, agent = make_system(RESERVOIR_CONFIG, RL_CONFIG)
history = train_agent(env, reservoir, agent, RL_CONFIG)
eval_returns = evaluate_agent(env, reservoir, agent, RL_CONFIG)
plot_training_history(history, eval_returns)


## Manual Experiments

Small edits that are useful while testing.


In [ ]:
# Stop background work:
# cartpole_workbench.stop()

# Train longer from the current weights:
# more_history = train_agent(env, reservoir, agent, RL_CONFIG, episodes=1000, eval_every=200)
# history["returns"] = np.concatenate([history["returns"], more_history["returns"]])
# history["eval_history"].extend(more_history["eval_history"])
# plot_training_history(history, evaluate_agent(env, reservoir, agent, RL_CONFIG))

# Rebuild after changing reservoir parameters:
# RESERVOIR_CONFIG = replace(RESERVOIR_CONFIG, n_reservoir=512, seed=5)
# env, reservoir, agent = make_system(RESERVOIR_CONFIG, RL_CONFIG)

# Change RL hyperparameters before rerunning training:
# RL_CONFIG = replace(RL_CONFIG, train_episodes=3000, actor_lr=5e-1, critic_lr=2e-2, gamma=0.97, batch_episodes=32, max_policy_logit_step=0.20, entropy_floor=0.45, ppo_clip=0.20, ppo_epochs=4, target_kl=0.03, line_search=True, line_search_backtracks=8)


# MPR-specific toggles:
# RESERVOIR_CONFIG = replace(RESERVOIR_CONFIG, randomize_cell_dynamics=False)
# RESERVOIR_CONFIG = replace(RESERVOIR_CONFIG, substeps_per_env_step=2, rate_feature_scale=2.0)
